# MobileNet Inference with TaskVine PythonTask

This notebook classifies a manifest-defined image dataset with ordinary
TaskVine `PythonTask` tasks. Each microbatch runs independently and creates a
new ONNX Runtime session. Floability stages the data and software environment
and launches the TaskVine workers.

For persistent worker-side model state, compare this notebook with
`mobilenet-serverless-taskvine.ipynb`.


## Step 1 — Configure the workload

The dataset is divided into deterministic microbatches. Every task returns the
top `TOP_K` predictions for each image in its batch.


In [ ]:
BATCH_SIZE = 4
TOP_K = 5


## Step 2 — Import helpers and locate staged inputs

Floability runs this notebook inside the staged `workflow/` directory. The
paths below refer to data that Floability copied or downloaded according to
`data/data.yml`.

The workflow-specific helper module is registered by value so TaskVine can
serialize its PythonTask function for worker execution.


In [ ]:
import json
import os
import shutil
import time
from pathlib import Path

import cloudpickle
from IPython.display import display
from PIL import Image

import mobilenet_helpers

cloudpickle.register_pickle_by_value(mobilenet_helpers)

if os.environ.get("FLOABILITY_WORKERS_ENABLED") == "0":
    raise RuntimeError("This PythonTask notebook requires Floability workers")

MODEL_PATH = Path("data/mobilenetv2-10.onnx")
LABELS_PATH = Path("data/imagenet-synset.txt")
IMAGE_DIR = Path("data/images")
MANIFEST_PATH = Path("data/image-manifest.json")
OUTPUT_DIR = Path("outputs")


## Step 3 — Validate the dataset contract and create batches

The manifest defines the dataset identity, image count, filenames, and SHA-256
checksums. The workflow validates it before any distributed task is submitted.
Sorting by filename and batching deterministically makes repeated runs
comparable.


In [ ]:
for required_path in (MODEL_PATH, LABELS_PATH):
    if not required_path.is_file():
        raise FileNotFoundError(f"Required staged input not found: {required_path}")

image_paths, manifest = mobilenet_helpers.load_and_verify_images(
    IMAGE_DIR,
    MANIFEST_PATH,
)
image_batches = mobilenet_helpers.make_image_batches(image_paths, BATCH_SIZE)
expected_image_names = [path.name for path in image_paths]

print(f"Dataset: {manifest['dataset_id']}@{manifest['dataset_version']}")
print(f"Verified images: {len(image_paths)}")
print(f"Microbatches: {len(image_batches)}")


## Step 4 — Create the TaskVine manager and declare inputs

Floability supplies a unique manager name and the permitted manager-port range.
The notebook creates the manager, materializes one directory per image batch,
and declares the model, labels, and batches as TaskVine inputs.


In [ ]:
import ndcctools.taskvine as vine

manager_name = os.environ.get("VINE_MANAGER_NAME")
if not manager_name:
    raise RuntimeError("VINE_MANAGER_NAME is not set; run through Floability")

port_spec = os.environ.get("VINE_MANAGER_PORTS", "9123,9150")
ports = [int(value.strip()) for value in port_spec.split(",") if value.strip()]
if not ports:
    raise ValueError("VINE_MANAGER_PORTS does not contain a port")
manager_port = ports[0] if len(ports) == 1 else [min(ports), max(ports)]

manager = vine.Manager(port=manager_port, name=manager_name)
batch_root, batch_paths = mobilenet_helpers.materialize_batch_directories(
    image_batches,
    prefix="mobilenet-python-task-batches-",
)

declared_model = manager.declare_file(str(MODEL_PATH), cache=True)
declared_labels = manager.declare_file(str(LABELS_PATH), cache=True)
declared_batches = {
    batch_path: manager.declare_file(str(batch_path), cache=True)
    for batch_path in batch_paths
}

print(f"Manager name: {manager_name}")
print(f"Manager port: {manager.port}")
print("Declared the model, labels, and image microbatches")


## Step 5 — Submit one PythonTask per batch

`classify_batch_with_new_session` is the worker function. Each PythonTask
receives the model and labels, loads a new inference session, classifies its
image batch, and returns predictions plus execution metadata.


In [ ]:
task_batches = {}
started_at = time.perf_counter()

for batch_path in batch_paths:
    task = vine.PythonTask(
        mobilenet_helpers.classify_batch_with_new_session,
        "model.onnx",
        "labels.txt",
        "batch",
        TOP_K,
    )
    task.add_input(declared_model, "model.onnx")
    task.add_input(declared_labels, "labels.txt")
    task.add_input(declared_batches[batch_path], "batch")
    task.set_cores(1)

    task_id = manager.submit(task)
    task_batches[task_id] = batch_path.name

print(f"Submitted {len(task_batches)} PythonTasks")


## Step 6 — Collect task results

Results may arrive in any order. The manager records which worker completed
each batch and fails the workflow if any task fails or returns an exception.


In [ ]:
results = []
failures = []

while not manager.empty():
    completed = manager.wait(5)
    if not completed:
        continue
    if not completed.successful():
        failures.append((completed.id, completed.result))
        print(f"FAILED task={completed.id} result={completed.result}")
        continue
    if isinstance(completed.output, Exception):
        failures.append((completed.id, repr(completed.output)))
        print(f"FAILED task={completed.id} exception={completed.output!r}")
        continue

    result = completed.output
    result["task_id"] = completed.id
    result["batch"] = task_batches[completed.id]
    result["worker_address"] = completed.addrport
    results.append(result)
    print(
        f"task={completed.id} batch={result['batch']} "
        f"session={result['session_load_id']} worker={completed.addrport}"
    )

if failures:
    raise RuntimeError(f"Inference task failures: {failures}")
if len(results) != len(batch_paths):
    raise RuntimeError(
        f"Expected {len(batch_paths)} task results; received {len(results)}"
    )

elapsed_seconds = time.perf_counter() - started_at
print(f"Execution completed in {elapsed_seconds:.2f} seconds")


## Step 7 — Validate and save the output

Every manifest image must appear exactly once. The summary records one session
load ID per PythonTask, while the contact sheet provides a compact visual check
of the classifications.


In [ ]:
predictions = mobilenet_helpers.predictions_by_image(
    results,
    expected_image_names,
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
summary = {
    "dataset_id": manifest["dataset_id"],
    "dataset_version": manifest["dataset_version"],
    "execution_mode": "python-task",
    "execution_mode_source": "mobilenet-python-task.ipynb",
    "image_count": len(image_paths),
    "batch_size": BATCH_SIZE,
    "batch_count": len(image_batches),
    "elapsed_seconds": elapsed_seconds,
    "distinct_library_loads": None,
    "task_results": results,
}
summary_path = OUTPUT_DIR / "python-task-summary.json"
summary_path.write_text(json.dumps(summary, indent=2) + "\n", encoding="utf-8")

contact_sheet_path = OUTPUT_DIR / "python-task-contact-sheet.jpg"
displayed_image_count = mobilenet_helpers.save_contact_sheet(
    image_paths,
    predictions,
    contact_sheet_path,
)

shutil.rmtree(batch_root)

print("=" * 72)
print("MOBILENET PYTHONTASK INFERENCE COMPLETE")
print(f"Validated images: {len(predictions)}")
print(f"Independent ONNX sessions: {len(results)}")
print(f"Elapsed time: {elapsed_seconds:.2f} seconds")
print(f"Results: {summary_path}")
print(f"Contact sheet: {contact_sheet_path} ({displayed_image_count} images shown)")
print("=" * 72)

display(Image.open(contact_sheet_path))


## Interpretation

PythonTask provides distributed task isolation and a direct Python function
interface. In this example, every microbatch pays the cost of creating its own
ONNX Runtime session. The serverless notebook demonstrates how a persistent
TaskVine library can reuse that initialized state.
